<a href="https://colab.research.google.com/github/AliAnusKhan/neurofive-ml-track/blob/Updated-Titanic-model-to-use-Scikit-Learn-Pipeline-and-new-features/ML_pipeline_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
file_path='/content/drive/MyDrive/dataset/train.csv'
titanic=pd.read_csv(file_path)


In [5]:
titanic.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Build a single Pipeline using ColumnTransformer that applies StandardScaler to numerical columns and OneHotEncoder to categorical columns

In [8]:
numerical_col=titanic.select_dtypes(include=['int64','float64']).columns
categorical_col=titanic.select_dtypes(include=['object']).columns
print("Numerical Columns:",numerical_col.tolist())
print("Categorical Columns:",categorical_col.tolist())
print("Numerical Columns:",len(numerical_col))
print("Categorical Columns:",len(categorical_col))


Numerical Columns: ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
Categorical Columns: ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']
Numerical Columns: 7
Categorical Columns: 5


In [9]:
titanic['FamilySize'] = titanic['SibSp'] + titanic['Parch'] + 1
titanic['IsAlone'] = (titanic['FamilySize']==1).astype(int)

In [10]:
y=titanic['Survived']
X=titanic[['Pclass','Sex','Age','Fare','Embarked','FamilySize','IsAlone']]

In [12]:
num_cols=['Age','Fare','FamilySize']
cat_cols=['Sex','Embarked','Pclass','IsAlone']

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

In [14]:
num_pipeline=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])
cat_pipeline=Pipeline(steps=[
    ('imputer',SimpleImputer(strategy='most_frequent')),
    ('onehot',OneHotEncoder())
])
preprocessor=ColumnTransformer(transformers=[
    ('num',num_pipeline,num_cols),
    ('cat',cat_pipeline,cat_cols)
])


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib

In [16]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [18]:
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        max_depth=5,
        min_samples_split=5,
        n_estimators=100,
        random_state=42
    ))
])

In [19]:
model_pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare',
                                                   'FamilySize']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder())]),
                                                  ['Sex', 'Embarked', 'Pclass',
                                                   'IsAlone'])])),
                ('classifier',
                 RandomForestClassifier(max_depth=5, min_samples_split=5,
                                        random_state=42))])

In [21]:
y_pred_pipe=model_pipeline.predict(X_test)

In [22]:
print("--- Pipeline Model Evaluation Report ---")
print(classification_report(y_test, y_pred_pipe))

joblib.dump(model_pipeline, 'titanic_pipeline.pkl')
print("\nPipeline successfully saved as 'titanic_pipeline.pkl'!")

--- Pipeline Model Evaluation Report ---
              precision    recall  f1-score   support

           0       0.81      0.89      0.85       105
           1       0.81      0.70      0.75        74

    accuracy                           0.81       179
   macro avg       0.81      0.79      0.80       179
weighted avg       0.81      0.81      0.81       179


Pipeline successfully saved as 'titanic_pipeline.pkl'!
